In [1]:
import numpy as np
import dendropy
from Bio import Phylo
import pandas as pd
from tqdm import tqdm
from src.utils.config import PHYLO_TREE, FINAL_LABELS, PHYLO_DIST_MATRIX, FINAL_LABELS_FILTERED, PROCESSED_DIR

In [2]:
tree = Phylo.read(PHYLO_TREE, "newick")
tree_species = set([tip.name for tip in tree.get_terminals()])

final_labels = pd.read_csv(FINAL_LABELS)

# Get species in your labels
label_species = set(final_labels['genome_id'])
print(f"Species in tree: {len(tree_species)}")
print(f"Species in labels: {len(label_species)}")

# Check overlap
overlap = sorted(tree_species & label_species)
missing_from_tree = label_species - tree_species
missing_from_labels = tree_species - label_species

print(f"\nOverlap: {len(overlap)} species")
print(f"In labels but NOT in tree: {len(missing_from_tree)}")
print(f"In tree but NOT in labels: {len(missing_from_labels)}")

if len(missing_from_tree) > 0:
    print(f"\nExample species missing from tree:")
    print(list(missing_from_tree)[:5])

Species in tree: 4716
Species in labels: 4744

Overlap: 4700 species
In labels but NOT in tree: 44
In tree but NOT in labels: 16

Example species missing from tree:
['MGYG000003365.1', 'MGYG000003888', 'MGYG000003531', 'MGYG000003605.1', 'MGYG000003782.1']


In [4]:
filtered_labels = final_labels[final_labels['genome_id'].isin(overlap)].copy()
filtered_labels = filtered_labels.set_index('genome_id').loc[overlap].reset_index()

# Load tree with dendropy for O(n²) single-pass distance computation
taxa = dendropy.TaxonNamespace()
dtree = dendropy.Tree.get(path=str(PHYLO_TREE), schema="newick", taxon_namespace=taxa)
pdm = dtree.phylogenetic_distance_matrix()

n_species = len(overlap)
distance_matrix = np.zeros((n_species, n_species))

for i, sp_A in tqdm(enumerate(overlap), total=n_species,
                    desc='Building Dist Matrix'):
    taxon_A = taxa.get_taxon(label=sp_A)
    for j in range(i + 1, n_species):
        taxon_B = taxa.get_taxon(label=overlap[j])
        dist = pdm(taxon_A, taxon_B)
        distance_matrix[i, j] = dist
        distance_matrix[j, i] = dist

print(f"Distance matrix shape: {distance_matrix.shape}")

Distance matrix shape: (4700, 4700)


In [5]:
# Save the distance matrix as numpy binary file (fast, compact)
np.save("/Users/encordsf/Desktop/phylo-gnn/data/processed/phylo_distance_matrix.npy", distance_matrix)

# Save the species ordering (so you know row 0 = which species)
species_order_df = pd.DataFrame({'genome_id': overlap})
species_order_df.to_csv('/Users/encordsf/Desktop/phylo-gnn/data/processed/phylo_species_order.csv', index=False)

# Save filtered labels (aligned to same ordering)
filtered_labels.to_csv(FINAL_LABELS_FILTERED, index=False)